# Dual-Head Classification — Hyperparameter Tuning
A single neural net with a **shared backbone** and **two output heads**:
- **Binary head**: 2-class Softmax → predicts 0 or 1 (disease absent/present)
- **Multi-class head**: 5-class Softmax → predicts severity 0–4

Loss = binary NLL + multi-class NLL (equal weighting, tunable via `loss_weight`).

Grid search over learning rate, hidden size, dropout, weight decay, and loss weighting.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from itertools import product
import matplotlib.pyplot as plt

## Load & Preprocess Data

In [ ]:
import ssl, urllib.request, io

url = "https://raw.githubusercontent.com/dataprofessor/data/master/heart-disease-cleveland.csv"

# macOS Python 3.10 sometimes has SSL cert issues — use unverified context as fallback
try:
    df = pd.read_csv(url)
except Exception:
    ctx = ssl._create_unverified_context()
    with urllib.request.urlopen(url, context=ctx) as r:
        df = pd.read_csv(io.BytesIO(r.read()))

# Strip leading spaces from column names
df.columns = df.columns.str.strip()

# Drop rows with missing values ('?')
df = df.replace('?', np.nan).dropna()

# Keep diagnosis as 0–4 (cast to int)
df['diagnosis'] = df['diagnosis'].astype(int)

# Binary label: clamp non-zero to 1
df['diagnosis_binary'] = df['diagnosis'].apply(lambda x: 1 if x > 0 else 0)

print('Multi-class distribution:')
print(df['diagnosis'].value_counts().sort_index())
print('\nBinary distribution:')
print(df['diagnosis_binary'].value_counts().sort_index())
df.head()

## Train / Validation / Test Split

In [ ]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

n = len(df)
train_end = int(0.7 * n)
val_end   = int(0.85 * n)

train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

feature_cols = [c for c in df.columns if c not in ('diagnosis', 'diagnosis_binary')]

def to_tensors(split):
    X      = torch.tensor(split[feature_cols].values.astype(np.float32))
    y_bin  = torch.tensor(split['diagnosis_binary'].values.astype(np.int64))
    y_multi= torch.tensor(split['diagnosis'].values.astype(np.int64))
    return X, y_bin, y_multi

X_train, y_bin_train, y_multi_train = to_tensors(train_df)
X_val,   y_bin_val,   y_multi_val   = to_tensors(val_df)
X_test,  y_bin_test,  y_multi_test  = to_tensors(test_df)

# Normalize using training stats
mean = X_train.mean(dim=0)
std  = X_train.std(dim=0) + 1e-8
X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std
X_test  = (X_test  - mean) / std

## Dual-Head Neural Network

In [ ]:
class DualHeadNet(nn.Module):
    """
    Shared backbone → two separate heads:
      - binary_head  : 2-class Softmax  (disease absent / present)
      - multi_head   : 5-class Softmax  (severity 0–4)
    """
    def __init__(self, input_dim, hidden_size, dropout):
        super().__init__()

        # Shared backbone
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Binary head: 2 outputs → Softmax
        self.binary_head = nn.Sequential(
            nn.Linear(hidden_size // 2, 2),
            nn.Softmax(dim=1)
        )

        # Multi-class head: 5 outputs → Softmax
        self.multi_head = nn.Sequential(
            nn.Linear(hidden_size // 2, 5),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        shared = self.backbone(x)
        return self.binary_head(shared), self.multi_head(shared)

## Training & Evaluation Helpers

In [ ]:
def train_model(model, X_tr, y_bin_tr, y_multi_tr,
                X_v,  y_bin_v,  y_multi_v,
                lr, weight_decay, loss_weight=0.5,
                epochs=150, batch_size=32):
    """
    loss_weight: weight for the binary head loss.
    Total loss = loss_weight * binary_loss + (1 - loss_weight) * multi_loss
    """
    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    dataset = TensorDataset(X_tr, y_bin_tr, y_multi_tr)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb_bin, yb_multi in loader:
            optimizer.zero_grad()
            out_bin, out_multi = model(xb)
            loss_bin   = criterion(torch.log(out_bin   + 1e-8), yb_bin)
            loss_multi = criterion(torch.log(out_multi + 1e-8), yb_multi)
            loss = loss_weight * loss_bin + (1 - loss_weight) * loss_multi
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        train_losses.append(epoch_loss / len(X_tr))

        model.eval()
        with torch.no_grad():
            vb, vm = model(X_v)
            v_loss = (loss_weight * criterion(torch.log(vb + 1e-8), y_bin_v)
                      + (1 - loss_weight) * criterion(torch.log(vm + 1e-8), y_multi_v)).item()
        val_losses.append(v_loss)

    return train_losses, val_losses


def accuracy_both(model, X, y_bin, y_multi):
    model.eval()
    with torch.no_grad():
        out_bin, out_multi = model(X)
        acc_bin   = (out_bin.argmax(dim=1)   == y_bin  ).float().mean().item()
        acc_multi = (out_multi.argmax(dim=1) == y_multi).float().mean().item()
    return acc_bin, acc_multi

## Hyperparameter Grid Search

In [ ]:
param_grid = {
    'lr':           [1e-2, 1e-3, 1e-4],
    'hidden_size':  [32, 64, 128],
    'dropout':      [0.0, 0.3],
    'weight_decay': [0.0, 1e-4],
    'loss_weight':  [0.3, 0.5, 0.7],  # binary vs multi-class loss balance
}

input_dim = X_train.shape[1]
EPOCHS    = 150

results = []

combos = list(product(*param_grid.values()))
print(f'Testing {len(combos)} hyperparameter combinations...')

for i, (lr, hidden_size, dropout, wd, lw) in enumerate(combos):
    torch.manual_seed(0)
    model = DualHeadNet(input_dim, hidden_size, dropout)
    train_losses, val_losses = train_model(
        model,
        X_train, y_bin_train, y_multi_train,
        X_val,   y_bin_val,   y_multi_val,
        lr=lr, weight_decay=wd, loss_weight=lw, epochs=EPOCHS
    )
    val_acc_bin, val_acc_multi = accuracy_both(model, X_val, y_bin_val, y_multi_val)
    # Sort by average of the two val accuracies
    combined = (val_acc_bin + val_acc_multi) / 2
    results.append({
        'lr': lr, 'hidden_size': hidden_size, 'dropout': dropout,
        'weight_decay': wd, 'loss_weight': lw,
        'val_acc_bin': val_acc_bin, 'val_acc_multi': val_acc_multi,
        'combined': combined,
        'final_val_loss': val_losses[-1],
        'model': model, 'train_losses': train_losses, 'val_losses': val_losses
    })
    if (i + 1) % 18 == 0:
        print(f'  {i+1}/{len(combos)} done')

results.sort(key=lambda r: -r['combined'])
print('\nTop 5 configurations (by avg val accuracy):')
for r in results[:5]:
    print(f"  lr={r['lr']}, hidden={r['hidden_size']}, dropout={r['dropout']}, "
          f"wd={r['weight_decay']}, lw={r['loss_weight']} "
          f"→ bin_acc={r['val_acc_bin']:.4f}, multi_acc={r['val_acc_multi']:.4f}")

## Best Model — Loss Curves & Test Accuracy

In [ ]:
best = results[0]
print(f"Best config: lr={best['lr']}, hidden={best['hidden_size']}, "
      f"dropout={best['dropout']}, wd={best['weight_decay']}, loss_weight={best['loss_weight']}")
print()

test_acc_bin, test_acc_multi = accuracy_both(
    best['model'], X_test, y_bin_test, y_multi_test
)
print(f"Val  accuracy — binary: {best['val_acc_bin']:.4f}  |  multi-class: {best['val_acc_multi']:.4f}")
print(f"Test accuracy — binary: {test_acc_bin:.4f}  |  multi-class: {test_acc_multi:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(best['train_losses'], label='Train Loss (combined)')
plt.plot(best['val_losses'],   label='Val Loss (combined)')
plt.xlabel('Epoch')
plt.ylabel('NLL Loss')
plt.title('Best Dual-Head Model — Training vs Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

## Head Accuracy Comparison Across All Configs (Top 30)

In [ ]:
top_n = min(30, len(results))
top   = results[:top_n]

labels      = [f"lr={r['lr']}\nh={r['hidden_size']}" for r in top]
bin_accs    = [r['val_acc_bin']   for r in top]
multi_accs  = [r['val_acc_multi'] for r in top]

x = np.arange(top_n)
width = 0.4

plt.figure(figsize=(16, 5))
plt.bar(x - width/2, bin_accs,   width, label='Binary head', color='steelblue')
plt.bar(x + width/2, multi_accs, width, label='Multi-class head', color='coral')
plt.xticks(x, labels, fontsize=6, rotation=45, ha='right')
plt.ylabel('Validation Accuracy')
plt.title(f'Top {top_n} Configs — Dual-Head Binary vs Multi-Class Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

## Effect of Loss Weight on Head Accuracy

In [ ]:
# Aggregate by loss_weight to see trade-off between heads
loss_weights = sorted(set(r['loss_weight'] for r in results))
avg_bin, avg_multi = [], []

for lw in loss_weights:
    group = [r for r in results if r['loss_weight'] == lw]
    avg_bin.append(np.mean([r['val_acc_bin']   for r in group]))
    avg_multi.append(np.mean([r['val_acc_multi'] for r in group]))

plt.figure(figsize=(6, 4))
plt.plot(loss_weights, avg_bin,   'o-', label='Binary head avg acc')
plt.plot(loss_weights, avg_multi, 's-', label='Multi-class head avg acc')
plt.xlabel('Binary Loss Weight')
plt.ylabel('Avg Validation Accuracy')
plt.title('Loss Weight Trade-off Between Heads')
plt.legend()
plt.tight_layout()
plt.show()